# Inference GPT-5.5 s krátkým promptem

Inference GPT-5.5 (OpenAI API) na 80 testovacích záznamech s krátkou variantou systémového promptu.

Výstup: `outputs/predictions_short/gpt_5_5_zs_short.jsonl`

## 1. Instalace knihoven

In [1]:
%%capture
!pip install --upgrade openai

## 2. Připojení Drive + nastavení cest

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/bp-misleading-cs'

TEST_DATA_PATH = f'{PROJECT_ROOT}/data/processed/test.jsonl'
SYSTEM_PROMPT_PATH = f'{PROJECT_ROOT}/prompts/system_prompt_v2_short.txt'  # ZMĚNA: krátký prompt
OUTPUT_DIR = f'{PROJECT_ROOT}/outputs/predictions_short'  # ZMĚNA: oddělený adresář
OUTPUT_PATH = f'{OUTPUT_DIR}/gpt_5_5_zs_short.jsonl'  # ZMĚNA: _short suffix

os.makedirs(OUTPUT_DIR, exist_ok=True)

for path in [TEST_DATA_PATH, SYSTEM_PROMPT_PATH]:
    assert os.path.exists(path), f'Chybí: {path}'
    print(f'{path}')
print(f'output: {OUTPUT_PATH}')

✅ /content/drive/MyDrive/bp-misleading-cs/data/processed/test.jsonl
✅ /content/drive/MyDrive/bp-misleading-cs/prompts/system_prompt_v2_short.txt
✅ output: /content/drive/MyDrive/bp-misleading-cs/outputs/predictions_short/gpt_5_5_zs_short.jsonl


## 3. Připojení k OpenAI API

In [4]:
from google.colab import userdata
from openai import OpenAI

api_key = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=api_key)

try:
    models = client.models.list()
    available_models = [m.id for m in models.data]
    gpt_5_models = [m for m in available_models if 'gpt-5' in m.lower()]
    print(f'API key funguje')
    print(f'Dostupné GPT-5 modely:')
    for m in sorted(gpt_5_models):
        print(f'   - {m}')
except Exception as e:
    print(f'❌ Chyba: {e}')
    print('Zkontroluj OPENAI_API_KEY v Colab Secrets')

✅ API key funguje
✅ Dostupné GPT-5 modely:
   - gpt-5
   - gpt-5-2025-08-07
   - gpt-5-chat-latest
   - gpt-5-codex
   - gpt-5-mini
   - gpt-5-mini-2025-08-07
   - gpt-5-nano
   - gpt-5-nano-2025-08-07
   - gpt-5-pro
   - gpt-5-pro-2025-10-06
   - gpt-5-search-api
   - gpt-5-search-api-2025-10-14
   - gpt-5.1
   - gpt-5.1-2025-11-13
   - gpt-5.1-chat-latest
   - gpt-5.1-codex
   - gpt-5.1-codex-max
   - gpt-5.1-codex-mini
   - gpt-5.2
   - gpt-5.2-2025-12-11
   - gpt-5.2-chat-latest
   - gpt-5.2-codex
   - gpt-5.2-pro
   - gpt-5.2-pro-2025-12-11
   - gpt-5.3-chat-latest
   - gpt-5.3-codex
   - gpt-5.4
   - gpt-5.4-2026-03-05
   - gpt-5.4-mini
   - gpt-5.4-mini-2026-03-17
   - gpt-5.4-nano
   - gpt-5.4-nano-2026-03-17
   - gpt-5.4-pro
   - gpt-5.4-pro-2026-03-05
   - gpt-5.5
   - gpt-5.5-2026-04-23
   - gpt-5.5-pro
   - gpt-5.5-pro-2026-04-23


## 4. Volba modelu

In [5]:
MODEL_NAME = 'gpt-5.5'

print(f'Použitý model: {MODEL_NAME}')
print(f'\nMyslíš si, že je to ten, který chceš? Pokud ne, uprav buňku a spusť znovu.')

Použitý model: gpt-5.5

Myslíš si, že je to ten, který chceš? Pokud ne, uprav buňku a spusť znovu.


## 5. Načtení test dat + system promptu

In [6]:
import json

# Načíst krátký system prompt
with open(SYSTEM_PROMPT_PATH, 'r', encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read().strip()

print(f'System prompt: {len(SYSTEM_PROMPT)} znaků (KRÁTKÁ verze)')

TRAIN_DATA_PATH = f'{PROJECT_ROOT}/data/processed/train_instruction.jsonl'
if os.path.exists(TRAIN_DATA_PATH):
    with open(TRAIN_DATA_PATH, 'r', encoding='utf-8') as f:
        first_train = json.loads(f.readline())
    if SYSTEM_PROMPT == first_train['instruction']:
        print('System prompt JE IDENTICKÝ s instruction v train_instruction.jsonl')
    else:
        print(f'Délky: prompt={len(SYSTEM_PROMPT)}, train_instruction={len(first_train["instruction"])}')
        print('POZOR: System prompt se liší od trénovací instrukce!')

test_records = []
with open(TEST_DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        test_records.append(json.loads(line.strip()))

print(f'\nTest set: {len(test_records)} záznamů')
print(f'\nUkázka prvního záznamu:')
print(f'  id: {test_records[0]["id"]}')
print(f'  header: {test_records[0]["header"]}')
print(f'  E1 gold: {test_records[0]["annotation"]["E1-misleading_header_model_final"]}')

System prompt: 1264 znaků (KRÁTKÁ verze)
✅ System prompt JE IDENTICKÝ s instruction v train_instruction.jsonl

Test set: 80 záznamů

Ukázka prvního záznamu:
  id: reflex_24eac431
  header: Tahá prezident za kratší konec Ústavy než Macinka? Divoká kohabitace poškozuje zájmy Česka
  E1 gold: POTENTIALLY_MISLEADING


## 6. Sanity check — 1 záznam na malém testu

In [7]:
test_record = test_records[0]
user_message = f'Titulek: "{test_record["header"]}"'

print(f'=== SANITY CHECK ===')
print(f'Titulek: {test_record["header"]}')
print(f'Gold E1: {test_record["annotation"]["E1-misleading_header_model_final"]}')
print()

try:
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user_message},
        ],
        response_format={'type': 'json_object'},  # JSON mode!

    )

    raw_output = response.choices[0].message.content
    parsed = json.loads(raw_output)

    print(f'Sanity check úspěšný')
    print(f'\nRaw output:')
    print(raw_output[:500])
    print(f'\nPredikované E1: {parsed.get("E1-misleading_header_model_final")}')

    # Token usage
    usage = response.usage
    print(f'\nToken usage:')
    print(f'  Prompt tokens: {usage.prompt_tokens}')
    print(f'  Completion tokens: {usage.completion_tokens}')
    print(f'  Cena per record (GPT-5.5 standard): ${usage.prompt_tokens * 2.50/1e6 + usage.completion_tokens * 15.00/1e6:.4f}')

except Exception as e:
    print(f'❌ Chyba: {e}')

=== SANITY CHECK ===
Titulek: Tahá prezident za kratší konec Ústavy než Macinka? Divoká kohabitace poškozuje zájmy Česka
Gold E1: POTENTIALLY_MISLEADING

✅ Sanity check úspěšný

Raw output:
{
  "A1-scope_of_missing_context": "HIGH",
  "A2-scope_of_missing_context_type": [
    "CAUSALITY",
    "SCOPE",
    "PROCESS_STAGE",
    "RELEVANCE_OF_ATTRIBUTE"
  ],
  "A3-quantification_present": "NONE",
  "B1-framing_present": "YES",
  "B2-framing_type": [
    "EMOTIONAL_LANGUAGE",
    "CAUSAL_SHORTCUT",
    "CLICKBAIT_STYLE",
    "PRESUPPOSITION"
  ],
  "B3-attribution_clarity": "MISSING",
  "C1-misinterpretation_risk": "MEDIUM",
  "C3-conspiracy_pattern": "NONE",
  "D1-sensitive_domain": "

Predikované E1: POTENTIALLY_MISLEADING

Token usage:
  Prompt tokens: 454
  Completion tokens: 546
  Cena per record (GPT-5.4 standard): $0.0093


## 7. Hlavní inference loop

In [8]:
import time
from datetime import datetime

def parse_json_safe(raw_output):
    """Pokus o parsing — JSON mode by měl vždy vrátit validní JSON, ale jistota."""
    try:
        return json.loads(raw_output), 'ok'
    except json.JSONDecodeError:
        return None, 'failed'

def call_gpt(record, retries=3):
    """Volání GPT s retry logikou pro síťové chyby."""
    user_message = f'Titulek: "{record["header"]}"'

    for attempt in range(retries):
        try:
            response = client.chat.completions.create(
                model=MODEL_NAME,
                messages=[
                    {'role': 'system', 'content': SYSTEM_PROMPT},
                    {'role': 'user', 'content': user_message},
                ],
                response_format={'type': 'json_object'},

            )
            return response, None
        except Exception as e:
            if attempt < retries - 1:
                wait_time = 2 ** attempt  # 1s, 2s, 4s
                print(f'   Chyba (pokus {attempt+1}/{retries}): {e}, čekám {wait_time}s...')
                time.sleep(wait_time)
            else:
                return None, str(e)

# Hlavní inference loop
print(f'{"="*60}')
print(f'KONFIGURACE: {MODEL_NAME}')
print(f'Output: {OUTPUT_PATH}')
print(f'Záznamů: {len(test_records)}')
print(f'{"="*60}\n')

start_time = datetime.now()
total_prompt_tokens = 0
total_completion_tokens = 0
parsing_stats = {'ok': 0, 'failed': 0, 'api_error': 0}

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f_out:
    for idx, record in enumerate(test_records, 1):
        # Volání API
        response, error = call_gpt(record)

        if response is None:
            # API chyba — zaloguj a pokračuj
            output_record = {
                'id': record['id'],
                'header': record['header'],
                'config': {MODEL_NAME},
                'raw_output': '',
                'parsed_output': None,
                'parsing_status': 'api_error',
                'error': error,
                'gold_annotation': record['annotation'],
            }
            parsing_stats['api_error'] += 1
        else:
            raw_output = response.choices[0].message.content
            parsed, status = parse_json_safe(raw_output)

            output_record = {
                'id': record['id'],
                'header': record['header'],
                'config': '{MODEL_NAME}',
                'raw_output': raw_output,
                'parsed_output': parsed,
                'parsing_status': status,
                'gold_annotation': record['annotation'],
            }
            parsing_stats[status] += 1
            total_prompt_tokens += response.usage.prompt_tokens
            total_completion_tokens += response.usage.completion_tokens

        f_out.write(json.dumps(output_record, ensure_ascii=False) + '\n')
        f_out.flush()  # průběžné ukládání

        # Progress každých 10 záznamů
        if idx % 10 == 0:
            elapsed = (datetime.now() - start_time).total_seconds()
            eta = elapsed / idx * (len(test_records) - idx)
            print(f'  [{idx}/{len(test_records)}] elapsed: {elapsed:.0f}s, ETA: {eta:.0f}s')

# Souhrn
elapsed_total = (datetime.now() - start_time).total_seconds()
print(f'\n{"="*60}')
print(f'  HOTOVO za {elapsed_total:.0f}s')
print(f'{"="*60}')
print(f'\nParsing stats:')
for status, count in parsing_stats.items():
    print(f'  {status}: {count} ({count/len(test_records)*100:.1f}%)')

print(f'\nToken usage:')
print(f'  Prompt tokens: {total_prompt_tokens:,}')
print(f'  Completion tokens: {total_completion_tokens:,}')

# Cena podle GPT-5.5 cennika (Standard režim)
cost_input = total_prompt_tokens * 2.50 / 1_000_000
cost_output = total_completion_tokens * 15.00 / 1_000_000
total_cost = cost_input + cost_output
print(f'\nReálné náklady (GPT-5.5 standard):')
print(f'  Vstupní tokeny: ${cost_input:.4f}')
print(f'  Výstupní tokeny: ${cost_output:.4f}')
print(f'  CELKEM: ${total_cost:.4f} (cca {total_cost * 23:.2f} Kč)')

KONFIGURACE: gpt-5.5
Output: /content/drive/MyDrive/bp-misleading-cs/outputs/predictions_short/gpt_5_5_zs_short.jsonl
Záznamů: 80

  [10/80] elapsed: 127s, ETA: 888s
  [20/80] elapsed: 237s, ETA: 711s
  [30/80] elapsed: 338s, ETA: 564s
  [40/80] elapsed: 506s, ETA: 506s
  [50/80] elapsed: 617s, ETA: 370s
  [60/80] elapsed: 792s, ETA: 264s
  [70/80] elapsed: 880s, ETA: 126s
  [80/80] elapsed: 981s, ETA: 0s

  HOTOVO za 981s

Parsing stats:
  ok: 80 (100.0%)
  failed: 0 (0.0%)
  api_error: 0 (0.0%)

Token usage:
  Prompt tokens: 36,037
  Completion tokens: 41,125

Reálné náklady (GPT-5.4 standard):
  Vstupní tokeny: $0.0901
  Výstupní tokeny: $0.6169
  CELKEM: $0.7070 (cca 16.26 Kč)


## 8. Quick check výsledků

In [9]:
from collections import Counter

# Načíst predikce zpět
predictions = []
with open(OUTPUT_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        predictions.append(json.loads(line.strip()))

# E1 distribuce predikcí vs gold
predicted_e1 = []
for p in predictions:
    if p['parsed_output']:
        predicted_e1.append(p['parsed_output'].get('E1-misleading_header_model_final', 'MISSING'))
    else:
        predicted_e1.append('PARSE_FAIL')

gold_e1 = [p['gold_annotation']['E1-misleading_header_model_final'] for p in predictions]

print('E1 distribuce v PREDIKCÍCH:')
for label, count in Counter(predicted_e1).most_common():
    print(f'  {label}: {count} ({count/len(predictions)*100:.1f}%)')

print('\nE1 distribuce v GOLD:')
for label, count in Counter(gold_e1).most_common():
    print(f'  {label}: {count} ({count/len(predictions)*100:.1f}%)')

# Naivní accuracy
correct = sum(1 for p, g in zip(predicted_e1, gold_e1) if p == g)
print(f'\nE1 accuracy: {correct}/{len(predictions)} = {correct/len(predictions)*100:.1f}%')
print('\nPro F1 macro a další metriky pust lokálně 04_evaluation.py')

E1 distribuce v PREDIKCÍCH:
  POTENTIALLY_MISLEADING: 33 (41.2%)
  NOT_MISLEADING: 31 (38.8%)
  MISLEADING: 16 (20.0%)

E1 distribuce v GOLD:
  NOT_MISLEADING: 34 (42.5%)
  POTENTIALLY_MISLEADING: 30 (37.5%)
  MISLEADING: 16 (20.0%)

E1 accuracy: 69/80 = 86.2%

Pro F1 macro a další metriky pust lokálně 04_evaluation.py
